# Lesson 16 lab — a tool-calling agent, four ways

Same agent, four implementations. The agent has **three tools**, runs a real **tool-calling loop**, maintains **multi-turn memory**, and emits **observability events**. Those are the four capabilities we want to compare — retrieval is just one of the tools.

| File                  | Framework    | Tool-calling loop                  | Memory model                         | Observability hook              |
|-----------------------|--------------|------------------------------------|--------------------------------------|---------------------------------|
| `agent_scratch.py`    | none         | explicit `while` + OpenAI schema   | summary + recent window (you wrote it)| `Trace.emit()` (your own)       |
| `agent_langchain.py`  | LangChain    | `bind_tools` + hand-written loop   | `InMemoryChatMessageHistory`         | `BaseCallbackHandler`           |
| `agent_llamaindex.py` | LlamaIndex   | `FunctionAgent.run(...)`           | `ChatMemoryBuffer` (token-budget)    | `Settings.callback_manager`     |
| `agent_langgraph.py`  | LangGraph    | `ToolNode` + `tools_condition`     | `State` dict + `SqliteSaver`         | every node emits a trace span   |

Shared across all four: `tools_shared.py` holds the three plain-Python tools (retrieval, office-hours lookup, grade calculator). Each framework wraps them in its own type system.

Prereq: `OPENAI_API_KEY` (or `OPENROUTER_API_KEY`) in your environment.

## 0 — The three shared tools

Read this first. Every framework version wraps exactly these functions.

In [ ]:
from tools_shared import search_syllabus, get_office_hours, compute_grade

print(search_syllabus('late homework'))
print(get_office_hours('Wednesday'))
print(compute_grade(project=85, homework=90, exam=75, participation=100))

## 1 — From scratch

Open `agent_scratch.py` while this runs. Notice:

- `Memory` is ~20 lines: summary + window, hand-compressed with one LLM call when it overflows.
- `run_turn` is ~25 lines: the standard OpenAI tool loop — call model, dispatch tool calls, append tool results, repeat until the model stops asking for tools.
- `Trace.emit` is the only observability we have. No vendor, no OTel — just structured events you own.

Count what you see in the live feed: how many `llm_end` events? How many `tool_start` events? You will need this number when comparing to the framework versions.

In [ ]:
from agent_scratch import demo as demo_scratch
demo_scratch()

## 2 — LangChain

Open `agent_langchain.py`. The loop is still there (see `run_turn`) — we wrote it by hand deliberately, to show what `bind_tools` gives you and what it doesn't. The savings: no custom schema code, no manual dispatch table, no tool JSON parsing, and `BaseCallbackHandler` subscribes to every LLM and tool event for free.

What you should notice in the output: the `[tool_start]` lines printed by the `AgentTrace` callback. That is the same hook LangSmith, Langfuse, and Phoenix subscribe to — you have just implemented the primitive they all build on.

In [ ]:
from agent_langchain import demo as demo_lc
demo_lc()

### 2a — `langchain.debug = True`: what the "one-liner" really did

Turn it on. The next invocation dumps every LLM prompt, every tool call, every intermediate step. You will use this every time you adopt a new LangChain primitive.

In [ ]:
import langchain
langchain.debug = True

from agent_langchain import run_turn, AgentTrace
tracer = AgentTrace()
_ = run_turn('debug-session', 'What are your office hours on Wednesday?', tracer)

langchain.debug = False

## 3 — LlamaIndex

Open `agent_llamaindex.py`. The agent is **built** in ~20 lines: wrap each function with `FunctionTool.from_defaults`, pass them to `FunctionAgent`, pair with a `ChatMemoryBuffer`. The tool loop and multi-turn context management are entirely inside the framework.

The trade: when you want to change how the agent decides between tools, or add a pre-tool guardrail, you are reading LlamaIndex source.

In [ ]:
from agent_llamaindex import demo as demo_li
demo_li()

## 4 — LangGraph

Open `agent_langgraph.py`. This is the shape LangChain now recommends for any non-trivial agent: explicit graph, explicit state, explicit checkpointer.

- `ToolNode(TOOLS)` runs all pending tool calls in parallel.
- `tools_condition` inspects the last AI message and routes back to `tools` if it contains `tool_calls`, else to `END`.
- `SqliteSaver` persists the state at every edge crossing. At the end we print the checkpoint history — that's your time-travel debugger.

Memory is no longer a special object; it's the `messages` field in `State`. To ship to production you swap the in-memory SQLite for `SqliteSaver.from_conn_string('file:agent.db?mode=rwc&uri=true')` or a Postgres/Redis checkpointer.

In [ ]:
from agent_langgraph import demo as demo_lg
demo_lg()

## 5 — Memory: the same conversation, four different persistence stories

Line by line, what you are trusting:

| Version     | State lives where                                   | Survives process restart? |
|-------------|-----------------------------------------------------|---------------------------|
| scratch     | `Memory.recent` list + `Memory.summary` string      | No, unless you add pickling |
| LangChain   | `_STORE[session_id] -> InMemoryChatMessageHistory`  | No. Swap for Redis / Postgres backends. |
| LlamaIndex  | `ChatMemoryBuffer` instance you passed to `agent.run` | No. Persist by saving `memory.chat_store`. |
| LangGraph   | SQLite row keyed by `thread_id` + `checkpoint_id`   | **Yes**, with `:memory:` replaced by a file. |

LangGraph's model is the only one that treats memory as first-class persisted state. That is its main value over LangChain for agentic systems.

## 6 — Observability: the same agent, three tracing backends

Every framework version above emits events through a callback interface. The backend is your choice, and you can switch by setting env vars — no code change.

**LangSmith** (vendor, tightly integrated with LangChain / LangGraph):
```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY=...
export LANGCHAIN_PROJECT="ai-design-lesson-16"
```

**Langfuse** (OSS, self-hostable, framework-agnostic):
```python
from langfuse.callback import CallbackHandler
tracer = CallbackHandler()
# pass tracer in config={"callbacks": [tracer]} wherever we passed AgentTrace
```

**OpenTelemetry + OpenLLMetry** (open standard, any OTLP backend):
```python
from traceloop.sdk import Traceloop
Traceloop.init(app_name='ai-design-lesson-16')   # auto-instruments LangChain and OpenAI
```

All three use the same underlying event stream. The teaching point: pick your observability layer before you pick your framework.

## 7 — Your turn (homework hook)

Pick one component of your running project (a planner, a tool router, a memory layer). Rewrite it with LangChain, LlamaIndex, **or** LangGraph. Keep both in git. Measure: lines of code, p50 and p95 latency, cost per call, and debuggability. Wire up LangSmith, Langfuse, or OTel tracing. Write a one-page decision memo.

There is no right answer — show your reasoning.